### how to use RNN cell

In [2]:
import torch

batch_size = 1
seq_len = 3    # 序列长度，例如一个单词有3个字母
input_size = 4    # 输入特征维度，例如一个字母有4个特征
hidden_size = 2    # 隐藏状态维度

# 构建RNNCell
cell = torch.nn.RNNCell(input_size=input_size, hidden_size=hidden_size)

# 形状为(seq, batch, feature)
dataset = torch.randn(seq_len, batch_size, input_size)
hidden = torch.zeros(batch_size, hidden_size)

# 循环计算cell, 依次读取序列中的每个元素, 计算得到隐藏状态作为下一个cell的输入
for idx, input in enumerate(dataset):
    print('=' * 20, idx, '=' * 20)
    print('input size: ', input.shape)
    hidden = cell(input, hidden)
    print('outputs size: ', hidden.shape)
    print(hidden)

==================== 0 ====================
input size:  torch.Size([1, 4])
outputs size:  torch.Size([1, 2])
tensor([[ 0.2756, -0.7336]], grad_fn=<TanhBackward0>)
==================== 1 ====================
input size:  torch.Size([1, 4])
outputs size:  torch.Size([1, 2])
tensor([[ 0.3051, -0.3017]], grad_fn=<TanhBackward0>)
==================== 2 ====================
input size:  torch.Size([1, 4])
outputs size:  torch.Size([1, 2])
tensor([[-0.7133, -0.9509]], grad_fn=<TanhBackward0>)


In [ ]:
# train a model to learn: hello -> ohlol
import torch
 
batch_size = 1
# seq_len = 3
input_size = 4
hidden_size = 4    # ohlol只有3个字母，为什么是4, 因为需要4个特征来表示每个字母, 即独热编码的每个元素的长度为4
# num_layers = 1
 
idx2char = ['e', 'h', 'l', 'o']    # 创建一个用来查询的列表, 每个字母对应一个索引
x_data = [1, 0, 2, 2, 3]    # hello
y_data = [3, 1, 2, 3, 2]    # ohlol

# 创建一个one-hot编码的查找表, 每个字母对应一个one-hot编码
one_hot_lookup = [[1, 0, 0, 0],
                  [0, 1, 0, 0],
                  [0, 0, 1, 0],
                  [0, 0, 0, 1]]
x_one_hot = [one_hot_lookup[x] for x in x_data]    # 将x_data中的每个字母转换为one-hot编码，h对应[0,1,0,0], 以此类推
inputs = torch.Tensor(x_one_hot).view(-1, batch_size, input_size)    # 将x_one_hot转换为张量，形状为(seq_len, batch_size, input_size)
labels = torch.LongTensor(y_data).view(-1, 1)    # 将y_data转换为张量，形状为(seq_len, 1)
 
 
class Model(torch.nn.Module):
    def __init__(self, input_size, hidden_size, batch_size):
        super(Model, self).__init__()
        #self.num_layers = num_layers
        self.batch_size = batch_size
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.rnncell = torch.nn.RNNCell(input_size=self.input_size,
                                        hidden_size=self.hidden_size)
 
    def forward(self, input, hidden):
        hidden = self.rnncell(input, hidden)
 
        return hidden
 
    def init_hidden(self):
        return torch.zeros(self.batch_size, self.hidden_size)

net = Model(input_size, hidden_size, batch_size)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)
 
for epoch in range(15):
    loss = 0
    optimizer.zero_grad()
    
    # 这里每次epoch都需要初始化hidden，是因为每个epoch是对整个序列的独立学习过程
    # 如果不初始化hidden，hidden会带着上一个epoch的状态，训练会出现串扰，不符合我们的设定
    # RNN的每个forward，输入是当前步的input和上一个time step的hidden，而不是上一个epoch的hidden
    hidden = net.init_hidden()
    print('Predicted string: ', end='')
    for input, label in zip(inputs, labels):    # zip函数将inputs和labels中的元素一一对应, 例如input[0]和label[0]对应, input[1]和label[1]对应, 以此类推
        hidden = net(input, hidden)
        loss += criterion(hidden, label)
        _, idx = hidden.max(dim=1)    # 取hidden中最大值的索引位置, _表示最大值, idx表示最大值的索引位置
        print(idx2char[idx.item()], end='')    # 打印出最大值的索引位置对应的字母
 
    loss.backward()    # 反向传播
    optimizer.step()    # 更新参数
    print(', Epoch [%d/15] loss = %.4f' % (epoch + 1, loss.item()))    # 打印出当前循环的损失值

Predicted string: lohlh, Epoch [1/15] loss = 6.5888
Predicted string: oolll, Epoch [2/15] loss = 5.6652
Predicted string: ollll, Epoch [3/15] loss = 5.0105
Predicted string: ohlll, Epoch [4/15] loss = 4.5407
Predicted string: ohlll, Epoch [5/15] loss = 4.1995
Predicted string: ohlll, Epoch [6/15] loss = 3.9199
Predicted string: ohlll, Epoch [7/15] loss = 3.6710
Predicted string: ohlol, Epoch [8/15] loss = 3.4396
Predicted string: ohlol, Epoch [9/15] loss = 3.2241
Predicted string: ohlol, Epoch [10/15] loss = 3.0451
Predicted string: ohlol, Epoch [11/15] loss = 2.9144
Predicted string: ohlol, Epoch [12/15] loss = 2.8045
Predicted string: ohlol, Epoch [13/15] loss = 2.6879
Predicted string: ohlol, Epoch [14/15] loss = 2.5709
Predicted string: ohlol, Epoch [15/15] loss = 2.4787


In [5]:
import torch
 
batch_size = 1
seq_len = 5
input_size = 4
hidden_size = 4
num_layers = 1
 
idx2char = ['e', 'h', 'l', 'o']
x_data = [1, 0, 2, 2, 3]
y_data = [3, 1, 2, 3, 2]

"""
inputs(seq_len 5,batch 1,input_size 4)
"""
one_hot_lookup = [[1, 0, 0, 0],
                  [0, 1, 0, 0],
                  [0, 0, 1, 0],
                  [0, 0, 0, 1]]
x_one_hot = [one_hot_lookup[x] for x in x_data]
inputs = torch.Tensor(x_one_hot).view(seq_len, batch_size, input_size)
labels = torch.LongTensor(y_data)
 
 
class Model(torch.nn.Module):
    def __init__(self, input_size, hidden_size, batch_size, num_layers=1):
        super(Model, self).__init__()
        self.num_layers = num_layers
        self.batch_size = batch_size
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.rnn = torch.nn.RNN(input_size=self.input_size,
                                hidden_size=self.hidden_size,
                                num_layers=num_layers)
 
    def forward(self, input):
        hidden = torch.zeros(self.num_layers,
                             self.batch_size,
                             self.hidden_size)
        out, _ = self.rnn(input, hidden)
 
        return out.view(-1, self.hidden_size)
 
net = Model(input_size, hidden_size, batch_size, num_layers)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.05)
 
for epoch in range(15):
    # loss = 0
    optimizer.zero_grad()
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    _, idx = outputs.max(dim=1)
    idx = idx.data.numpy()
    print('Predicted: ', ''.join([idx2char[x] for x in idx]), end='')
    print(', Epoch [%d/15] loss = %.3f' % (epoch + 1, loss.item()))

Predicted:  llhlo, Epoch [1/15] loss = 1.383
Predicted:  lllll, Epoch [2/15] loss = 1.273
Predicted:  lllll, Epoch [3/15] loss = 1.197
Predicted:  lllll, Epoch [4/15] loss = 1.145
Predicted:  lllll, Epoch [5/15] loss = 1.103
Predicted:  lllll, Epoch [6/15] loss = 1.064
Predicted:  lllll, Epoch [7/15] loss = 1.024
Predicted:  ollll, Epoch [8/15] loss = 0.983
Predicted:  ololl, Epoch [9/15] loss = 0.945
Predicted:  ololl, Epoch [10/15] loss = 0.916
Predicted:  oholl, Epoch [11/15] loss = 0.895
Predicted:  oholl, Epoch [12/15] loss = 0.874
Predicted:  oholl, Epoch [13/15] loss = 0.850
Predicted:  oholl, Epoch [14/15] loss = 0.827
Predicted:  oholl, Epoch [15/15] loss = 0.806


In [ ]:
# 使用embedding, 并且num_layers=2, 两层堆叠RNN
import torch
 
num_class = 4
batch_size = 1
seq_len = 5
input_size = 4
hidden_size = 8
num_layers = 2
embedding_size = 10

idx2char = ['e', 'h', 'l', 'o']
x_data = [[1, 0, 2, 2, 3]]
y_data = [3, 1, 2, 3, 2]
inputs = torch.LongTensor(x_data)
labels = torch.LongTensor(y_data)
 
 
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.emb = torch.nn.Embedding(input_size, embedding_size)
        self.rnn = torch.nn.RNN(input_size=embedding_size,
                                hidden_size=hidden_size,
                                num_layers=num_layers,
                                batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, num_class)    # 创建一个线性层，将hidden_size变成num_class，num_class为4
 
    def forward(self, x):
        hidden = torch.zeros(num_layers, x.size(0), hidden_size)
        x = self.emb(x)    # 将输入数据x进行嵌入，将input_size变成embedding_size(4-10)每一个数据4维变成10维
        x, _ = self.rnn(x, hidden)
        x = self.fc(x)
 
        return x.view(-1, num_class)

net = Model()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.05)
 
for epoch in range(15):
    # loss = 0
    optimizer.zero_grad()
    outputs = net(inputs)
    print(outputs.shape)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
 
    _, idx = outputs.max(dim=1)
    idx = idx.data.numpy()
 
    print('Predicted: ', ''.join([idx2char[x] for x in idx]), end='')
    print(', Epoch [%d/15] loss = %.3f' % (epoch + 1, loss.item()))

torch.Size([5, 4])
Predicted:  lllll, Epoch [1/15] loss = 1.403
torch.Size([5, 4])
Predicted:  ehlll, Epoch [2/15] loss = 1.081
torch.Size([5, 4])
Predicted:  ohlll, Epoch [3/15] loss = 0.894
torch.Size([5, 4])
Predicted:  ohlll, Epoch [4/15] loss = 0.727
torch.Size([5, 4])
Predicted:  ohlol, Epoch [5/15] loss = 0.553
torch.Size([5, 4])
Predicted:  ohlol, Epoch [6/15] loss = 0.405
torch.Size([5, 4])
Predicted:  ohlol, Epoch [7/15] loss = 0.293
torch.Size([5, 4])
Predicted:  ohlol, Epoch [8/15] loss = 0.214
torch.Size([5, 4])
Predicted:  ohlol, Epoch [9/15] loss = 0.147
torch.Size([5, 4])
Predicted:  ohlol, Epoch [10/15] loss = 0.098
torch.Size([5, 4])
Predicted:  ohlol, Epoch [11/15] loss = 0.066
torch.Size([5, 4])
Predicted:  ohlol, Epoch [12/15] loss = 0.049
torch.Size([5, 4])
Predicted:  ohlol, Epoch [13/15] loss = 0.034
torch.Size([5, 4])
Predicted:  ohlol, Epoch [14/15] loss = 0.023
torch.Size([5, 4])
Predicted:  ohlol, Epoch [15/15] loss = 0.017
